# Type I & Type II Errors
**Topic:** Inferential Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Define** Type I error (false positive) and Type II error (false negative) in terms of hypothesis testing
- **Explain** the trade-off between α and β and why reducing one increases the other
- **Interpret** statistical power (1 − β) as the probability of correctly detecting a real effect

> **Tip:** Start by moving the **effect size slider** toward zero, watch the two distributions overlap more and more, and see how power drops as the effect becomes harder to distinguish from noise.

---
## How we got here

In *17: Hypothesis Testing* and *18: P-Values* we made a binary decision: reject H₀ or fail to reject. But any binary decision under uncertainty has two failure modes. This notebook names them, quantifies them, and shows the fundamental tension between them that every statistician, data scientist, and decision-maker must navigate.

---
## Why this matters for data science

The Type I / Type II error framework is the statistical language of real-world cost-benefit analysis. In spam filtering, a Type I error (flagging real mail as spam) may be more costly than a Type II error (missing spam). In cancer screening, the opposite is true. In ML model evaluation, the precision-recall trade-off is exactly this framework, the ROC curve is a visual map of the boundary between the two error types.

---
## Try it yourself

In [2]:
from tkh_utils import make_slider, make_int_slider, make_output, display_widget, register_observer

out = make_output()
delta_slider = make_slider(0.5, 4.0, 0.1, 2.0, "Effect size (δ):")
alpha_slider = make_slider(0.01, 0.20, 0.01, 0.05, "Alpha (α):")
n_slider     = make_int_slider(5, 200, 5, 30, "Sample size (n):")

def render(delta, alpha, n):
    se        = 1.0 / np.sqrt(n)
    z_crit_std = stats.norm.ppf(1 - alpha)
    threshold  = z_crit_std * se

    span   = max(4 * se, abs(delta) + 4 * se)
    x_lo   = min(-span, delta - span)
    x_hi   = max(span, delta + span)
    x_rng  = np.linspace(x_lo, x_hi, 600)

    y_h0 = stats.norm.pdf(x_rng, 0, se)
    y_h1 = stats.norm.pdf(x_rng, delta, se)

    x_alpha = x_rng[x_rng >= threshold]
    y_alpha = stats.norm.pdf(x_alpha, 0, se)

    x_beta  = x_rng[x_rng <= threshold]
    y_beta  = stats.norm.pdf(x_beta, delta, se)
    beta_val = stats.norm.cdf(threshold, delta, se)
    power    = 1 - beta_val

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_rng, y=y_h0,
        mode="lines", line=dict(color=PALETTE["primary"], width=3),
        name="H₀ distribution",
    ))
    fig.add_trace(go.Scatter(
        x=x_rng, y=y_h1,
        mode="lines", line=dict(color=PALETTE["secondary"], width=3),
        name="H₁ distribution",
    ))
    if len(x_alpha) > 1:
        fig.add_trace(go.Scatter(
            x=np.concatenate([[x_alpha[0]], x_alpha, [x_alpha[-1]]]),
            y=np.concatenate([[0], y_alpha, [0]]),
            fill="toself", fillcolor="rgba(247,103,7,0.35)",
            line=dict(width=0),
            name=f"Type I error (α = {alpha:.2f})",
        ))
    if len(x_beta) > 1:
        fig.add_trace(go.Scatter(
            x=np.concatenate([[x_beta[0]], x_beta, [x_beta[-1]]]),
            y=np.concatenate([[0], y_beta, [0]]),
            fill="toself", fillcolor="rgba(76,110,245,0.30)",
            line=dict(width=0),
            name=f"Type II error (β = {beta_val:.3f})",
        ))
    fig.add_vline(
        x=threshold,
        line_color=PALETTE["accent"], line_width=2,
        annotation_text=f"Threshold = {threshold:.2f}",
    )
    layout = base_layout(
        title=f"Type I (α={alpha:.2f})  Type II (β={beta_val:.3f})  Power={power:.3f}",
        xaxis_title="Test Statistic",
        yaxis_title="Probability Density",
    )
    fig.update_layout(layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

register_observer(
    [delta_slider, alpha_slider, n_slider],
    lambda change: render(delta_slider.value, alpha_slider.value, n_slider.value),
)
display_widget([delta_slider, alpha_slider, n_slider], out)
render(delta_slider.value, alpha_slider.value, n_slider.value)


---
## What's happening?

Every hypothesis test can produce two kinds of errors, rejecting a true H₀ or failing to reject a false H₀.

| Decision | H₀ is actually TRUE | H₀ is actually FALSE |
|----------|--------------------|--------------------|
| Reject H₀ | **Type I Error** (False Positive): rate = α | Correct: rate = Power (1−β) |
| Fail to reject H₀ | Correct: rate = 1−α | **Type II Error** (False Negative): rate = β |

Power formula (for a one-sided z-test):

$$\text{Power} = P\!\left(Z > z_{\alpha} - \frac{\delta}{\sigma/\sqrt{n}}\right)$$

where δ is the true effect size.

### The fundamental trade-off

Decreasing α (being stricter about rejecting H₀) moves the rejection boundary outward, which automatically increases β, the overlap region where a real effect gets missed. The only way to reduce **both** simultaneously is to increase sample size n.

Look back at the widget: with effect size fixed, increase n and watch both the rejection region become well-separated from H₁ and power approach 1.

---
## Real-world example: COVID-19 rapid antigen test accuracy

Rapid antigen tests for COVID-19 illustrate both error types with real consequences. A false negative (Type II error) means an infected person tests negative and might spread the virus. A false positive (Type I error) means a healthy person tests positive and quarantines unnecessarily.

The chart visualizes the test score distributions for infected and healthy populations, with a decision threshold that generates both error types. Notice:

- **Notice:** The two distributions overlap, no threshold perfectly separates them; any decision line creates errors on both sides
- **Notice:** Moving the threshold left (more sensitive) grows the false-positive region and shrinks the false-negative region
- **Notice:** The area shaded on the left (β, Type II) plus the area shaded on the right (α, Type I) sum to the total error rate

> **Discussion question:** For a voluntary screening program with limited quarantine costs, would you set a more sensitive (lower threshold) or more specific (higher threshold) test, and who bears the cost of each type of error?

In [3]:
# ── Type I vs Type II error visualization for diagnostic test ─────────────────
# Hypothetical test score distributions for infected vs healthy individuals
mu_H0    = 0.0    # healthy population test score mean
mu_H1    = 2.5    # infected population test score mean
sigma    = 1.0
alpha    = 0.05
threshold = stats.norm.ppf(1 - alpha, loc=mu_H0, scale=sigma)

x_range  = np.linspace(-4, 6, 500)
pdf_H0   = stats.norm.pdf(x_range, mu_H0, sigma)
pdf_H1   = stats.norm.pdf(x_range, mu_H1, sigma)

# Type I region: H0 distribution above threshold
x_alpha  = x_range[x_range >= threshold]
# Type II region: H1 distribution below threshold
x_beta   = x_range[x_range <= threshold]
beta_val = stats.norm.cdf(threshold, mu_H1, sigma)
power    = 1 - beta_val

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_range, y=pdf_H0, mode="lines",
    line=dict(color=PALETTE["muted"], width=2.5), name="H₀ (Healthy)"))
fig.add_trace(go.Scatter(x=x_range, y=pdf_H1, mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5), name="H₁ (Infected)"))

# Type I shading
fig.add_trace(go.Scatter(
    x=np.concatenate([[threshold], x_alpha, [x_alpha[-1]]]),
    y=np.concatenate([[0], stats.norm.pdf(x_alpha, mu_H0, sigma), [0]]),
    fill="toself", fillcolor="rgba(247,103,7,0.30)",
    line=dict(width=0), name=f"Type I Error (α = {alpha:.2f})",
))
# Type II shading
fig.add_trace(go.Scatter(
    x=np.concatenate([[x_beta[0]], x_beta, [threshold]]),
    y=np.concatenate([[0], stats.norm.pdf(x_beta, mu_H1, sigma), [0]]),
    fill="toself", fillcolor="rgba(76,110,245,0.25)",
    line=dict(width=0), name=f"Type II Error (β = {beta_val:.2f})",
))
fig.add_vline(x=threshold, line_color=PALETTE["accent"], line_width=2,
              annotation_text=f"Threshold = {threshold:.2f}")
layout = base_layout(
    title=f"Type I (α={alpha:.2f}) & Type II (β={beta_val:.2f}) Errors  |  Power = {power:.2f}",
    xaxis_title="Test Score",
    yaxis_title="Probability Density",
)
fig.update_layout(layout)
fig.show()

### The cost of each error type across domains

| Domain | Type I Error (False Positive) | Type II Error (False Negative) | Which is worse? |
|--------|------------------------------|-------------------------------|----------------|
| Spam filter | Legitimate email flagged as spam | Spam reaches inbox | Depends on user; usually Type I |
| Cancer screening | Healthy patient undergoes biopsy | Cancer missed | Type II (missed cancer) |
| Fraud detection | Legitimate transaction blocked | Fraud transaction approved | Context-dependent |
| Drug approval | Ineffective drug approved | Effective drug rejected | Usually Type II |
| A/B test | Change deployed with no real effect | Real improvement goes undetected | Usually Type II |

---
## Key takeaway

> **Type I errors are false alarms at rate α; Type II errors are missed detections at rate β — reducing one always costs the other, and only larger samples can shrink both simultaneously.**

---
*Next up: T-Tests — the most common statistical test in practice, used when the population standard deviation is unknown and sample sizes are small*